# 🌐 Assistente Diversa — Educação Inclusiva
## Proposta final - TCC

Grupo: Hélice

Participantes:
- Brena Oliveira
- Cesar Ricciarelli
- João Pedro Maranhão
- Leandro Nascimento
- Nícolas Pinotti

## 1. Instalação de dependências



In [ ]:
%pip install -U beautifulsoup4 gradio pandas scikit-learn requests

print("✅ Instalação concluída!")
print()
print("Bibliotecas usadas no projeto:")
print("  • gradio     — interface web do chatbot")
print("  • requests   — chamadas HTTP para a Groq")
print("  • pandas     — manipulação da base de dados")
print("  • sklearn    — TF-IDF e busca por similaridade")


## 2. Importações e configuração



Carregamento da chave de API do GROQ devidamente configurada com proteção através do .env (exluso na configuração de .gitignore).

In [ ]:
import json
import os
import re
import time
from typing import Dict, List

import gradio as gr
import pandas as pd
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from dotenv import load_dotenv

load_dotenv()
GROQ_KEY = os.getenv("GROQ_KEY", "")

GROQ_MODEL = "llama-3.1-8b-instant"
GROQ_URL = "https://api.groq.com/openai/v1/chat/completions"

PARAMS_GERACAO = {
    "model": GROQ_MODEL,
    "max_tokens": 600,
    "temperature": 0.3,
    "top_p": 0.9,
}

print("✅ Configuração carregada")
print("Modelo:", GROQ_MODEL)
print("Modo:", "Groq/Llama" if GROQ_KEY else "Demonstração sem API")

## 3. Base de conhecimento


Os dados se encontram hospedados no google docs, onde os registros ficam mais seguros e o código fica mais limpo.

In [ ]:
import pandas as pd

SHEET_ID = "1Ll0gKNxMs20rMAESmjk4Dk2_FAtugKgY5NUhwaod1rI" 
URL_PLANILHA = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv"

# 1. Transforma a planilha em um Pandas DataFrame
artigos_brutos = pd.read_csv(URL_PLANILHA)

# 2. Visualizando dimensões do Dataframe
print("Linhas x Colunas =", artigos_brutos.shape)


# 3. Imprime as colunas (corrigido: sem os parênteses no final)
print("\nColunas presentes na base de dados:")
print(list(artigos_brutos.columns))
print()

# 4. Visualiza as 5 primeiras linhas de forma formatada no Jupyter
print("5 primeiras linhas")
display(artigos_brutos.head())

# 5. Visualização dos tipos dos objetos
print(f"Todos os objetos são de texto:")
display(artigos_brutos.dtypes)


artigos_brutos = [];

for titulo, url, texto, artigo_completo in artigos_brutos.values:
  artigos_brutos.append({"titulo": titulo,
        "url": url,
        "texto": artigo_completo})

## 4. Limpeza da base



In [ ]:
def limpar_texto(texto):
    texto = re.sub(r"\s+", " ", str(texto))
    texto = re.sub(r"\[.*?\]", "", texto)
    texto = re.sub(r"http\S+", "", texto)
    return texto.strip()

df = pd.DataFrame(artigos_brutos)
df["texto_limpo"] = df["texto"].apply(limpar_texto)
df = df[df["texto_limpo"].str.len() > 120].reset_index(drop=True)
df["conteudo_busca"] = df["titulo"] + ". " + df["titulo"] + ". " + df["texto_limpo"]

print("✅ Limpeza concluída")
print("Total de artigos válidos:", len(df))
display(df[["titulo", "url"]])

## 5. Vetorização TF-IDF e busca semântica


In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=8000,
    sublinear_tf=True,
    strip_accents="unicode",
    lowercase=True,
)

matriz_tfidf = vectorizer.fit_transform(df["conteudo_busca"])

def buscar_artigos_relevantes(pergunta, top_k=3, score_minimo=0.015):
    vetor = vectorizer.transform([pergunta])
    scores = cosine_similarity(vetor, matriz_tfidf)[0]
    indices = scores.argsort()[::-1][:top_k]

    resultados = []
    for i in indices:
        score = float(scores[i])
        if score >= score_minimo:
            resultados.append({
                "titulo": df.iloc[i]["titulo"],
                "url": df.iloc[i]["url"],
                "trecho": df.iloc[i]["texto_limpo"][:900],
                "score": round(score, 4),
            })
    return resultados

print("✅ Vetorização concluída")
print("Matriz:", matriz_tfidf.shape)

## 6. Teste da busca

In [ ]:
pergunta_teste = "O que é Atendimento Educacional Especializado?"
resultados = buscar_artigos_relevantes(pergunta_teste)

print("Pergunta:", pergunta_teste)
print("Artigos recuperados:")
for r in resultados:
        print(f"- {r['titulo']} | score={r['score']}")

## 7. Prompt do assistente e controle de escopo


In [ ]:
# ── System Prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT_PROFESSOR = (
    "Você é um assistente especialista em Educação Inclusiva do Portal Diversa (diversa.org.br), "
    "respondendo a um PROFESSOR da Educação Básica ou do AEE.\n\n"
    "Seu papel é oferecer orientações pedagógicas detalhadas e aplicáveis em sala de aula, "
    "com base nos conteúdos do Portal Diversa.\n\n"
    "ESTILO DE RESPOSTA:\n"
    "- Use linguagem técnica-pedagógica, mas didática\n"
    "- Estruture a resposta em passos práticos sempre que possível\n"
    "- Explique o 'porquê' pedagógico das recomendações, não só o 'o quê'\n"
    "- Pode usar termos técnicos (AEE, DUA, adaptação curricular), mas explique-os brevemente\n\n"
    "RESPONDA perguntas sobre:\n"
    "- Deficiências e transtornos: autismo (TEA), TDAH, deficiência visual, auditiva, intelectual e física\n"
    "- Legislação: LBI, LDB, BNCC, Política Nacional de Educação Especial\n"
    "- Estratégias pedagógicas inclusivas e adaptações curriculares\n"
    "- Atendimento Educacional Especializado (AEE)\n"
    "- Tecnologia assistiva\n\n"
    "NAO RESPONDA (redirecione educadamente):\n"
    "- Diagnósticos médicos ou psicológicos\n"
    "- Opiniões políticas ou partidárias\n"
    "- Assuntos sem relação com educação inclusiva\n\n"
    "Se a pergunta estiver fora do escopo, responda exatamente:\n"
    "'Minha especialidade é Educação Inclusiva. Posso te ajudar com alguma dúvida sobre esse tema?'\n\n"
    "Responda sempre em português brasileiro. Cite o título e a URL do artigo consultado ao final."
)

SYSTEM_PROMPT_FAMILIA = (
    "Você é um assistente especialista em Educação Inclusiva do Portal Diversa (diversa.org.br), "
    "respondendo a um FAMILIAR de uma pessoa com deficiência.\n\n"
    "Seu papel é acolher, esclarecer dúvidas e orientar sobre direitos e recursos disponíveis, "
    "com base nos conteúdos do Portal Diversa.\n\n"
    "ESTILO DE RESPOSTA:\n"
    "- Use linguagem simples, calorosa e acessível, evitando jargão técnico\n"
    "- Quando precisar usar um termo técnico (ex: AEE, LBI), explique-o em uma frase simples\n"
    "- Demonstre empatia genuína, especialmente em temas sensíveis (diagnóstico, bullying, direitos)\n"
    "- Seja direto e prático: a família quer saber o que fazer, não uma aula teórica\n\n"
    "RESPONDA perguntas sobre:\n"
    "- Deficiências e transtornos: autismo (TEA), TDAH, deficiência visual, auditiva, intelectual e física\n"
    "- Direitos garantidos por lei (LBI, PNEEI)\n"
    "- Como a escola deve apoiar a família e o estudante\n"
    "- Tecnologia assistiva e recursos disponíveis\n\n"
    "NAO RESPONDA (redirecione educadamente):\n"
    "- Diagnósticos médicos ou psicológicos — sempre recomende um profissional habilitado\n"
    "- Opiniões políticas ou partidárias\n"
    "- Assuntos sem relação com educação inclusiva\n\n"
    "Se a pergunta estiver fora do escopo, responda exatamente:\n"
    "'Minha especialidade é Educação Inclusiva. Posso te ajudar com alguma dúvida sobre esse tema?'\n\n"
    "Responda sempre em português brasileiro, de forma acolhedora. Cite a fonte ao final, "
    "de forma simples (ex: 'Você pode ler mais sobre isso aqui: [link]')."
)

SYSTEM_PROMPT_GESTOR = (
    "Você é um assistente especialista em Educação Inclusiva do Portal Diversa (diversa.org.br), "
    "respondendo a um GESTOR escolar ou profissional de secretaria de educação.\n\n"
    "Seu papel é fornecer informações objetivas sobre legislação, políticas e gestão de recursos "
    "para inclusão, com base nos conteúdos do Portal Diversa.\n\n"
    "ESTILO DE RESPOSTA:\n"
    "- Use linguagem técnica, objetiva e direta\n"
    "- Priorize dados concretos: nomes de leis, decretos, percentuais, prazos quando disponíveis\n"
    "- Evite explicações longas de conceitos básicos — assuma conhecimento prévio do gestor\n"
    "- Estruture em tópicos quando a resposta envolver múltiplos itens (normas, responsabilidades)\n\n"
    "RESPONDA perguntas sobre:\n"
    "- Legislação: LBI, LDB, BNCC, Decreto 12.686/2025 (PNEEI), PNEEPEI\n"
    "- Gestão escolar inclusiva: projeto político-pedagógico, formação de professores\n"
    "- Atendimento Educacional Especializado (AEE) e Salas de Recursos Multifuncionais\n"
    "- Políticas públicas e estrutura de rede de ensino\n\n"
    "NAO RESPONDA (redirecione educadamente):\n"
    "- Diagnósticos médicos ou psicológicos\n"
    "- Decisões judiciais específicas de casos individuais — oriente buscar assessoria jurídica\n"
    "- Opiniões políticas ou partidárias\n\n"
    "Se a pergunta estiver fora do escopo, responda exatamente:\n"
    "'Minha especialidade é Educação Inclusiva. Posso te ajudar com alguma dúvida sobre esse tema?'\n\n"
    "Responda sempre em português brasileiro, de forma objetiva. Cite a fonte legal/documental ao final."
)

SYSTEM_PROMPT = SYSTEM_PROMPT_PROFESSOR  # compatibilidade com código antigo

print("✅ System prompt configurado!")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Função: chamar_groq
# Envia mensagens para o Llama 3.1 via Groq com retry automático.
# Parâmetros: messages (list), tentativas (int)
# Retorna   : str — texto gerado ou mensagem de erro
# ─────────────────────────────────────────────────────────────────────────────
def chamar_groq(messages, tentativas=3):
    payload = {**PARAMS_GERACAO, "messages": messages}

    for tentativa in range(1, tentativas + 1):
        try:
            resp = requests.post(GROQ_URL, headers=HEADERS, json=payload, timeout=30)

            if resp.status_code == 200:
                return resp.json()["choices"][0]["message"]["content"].strip()

            elif resp.status_code == 429:
                # Rate limit — aguarda antes de tentar novamente
                espera = tentativa * 20
                print(f"  ⚠️  Rate limit (tentativa {tentativa}/{tentativas}). Aguardando {espera}s...")
                time.sleep(espera)

            elif resp.status_code == 401:
                return "❌ Chave inválida. Verifique o GROQ_KEY na Seção 3."

            else:
                return f"❌ Erro HTTP {resp.status_code}: {resp.text[:200]}"

        except requests.exceptions.Timeout:
            print(f"  ⏱️  Timeout na tentativa {tentativa}/{tentativas}")
            if tentativa < tentativas:
                time.sleep(10)
        except Exception as e:
            return f"❌ Erro inesperado: {str(e)}"

    return "⏳ Serviço indisponível. Tente novamente em instantes."


# ─────────────────────────────────────────────────────────────────────────────
# Função: responder
# Pipeline RAG completo: busca → monta mensagens → Groq → resposta com fontes.
# Parâmetro : pergunta (str)
# Retorna   : str — resposta com referências
# ─────────────────────────────────────────────────────────────────────────────
PERFIS = {
    "Professor": SYSTEM_PROMPT_PROFESSOR,
    "Família": SYSTEM_PROMPT_FAMILIA,
    "Gestor": SYSTEM_PROMPT_GESTOR,
}

def responder(pergunta, historico=None, perfil="Professor"):
    if not pergunta.strip():
        return "Por favor, digite sua pergunta sobre Educação Inclusiva."

    # .get() com fallback pra Professor caso o valor não bata com as chaves
    system_prompt_escolhido = PERFIS.get(perfil, SYSTEM_PROMPT_PROFESSOR)

    # Passo 1: buscar artigos relevantes na base indexada
    artigos = buscar_artigos_relevantes(pergunta, top_k=3)

    # Passo 2: montar contexto com os trechos encontrados
    if artigos:
        contexto = "\n\n".join([
            f"Artigo {i+1}: {a['titulo']}\nTrecho: {a['trecho']}\nFonte: {a['url']}"
            for i, a in enumerate(artigos)
        ])
        instrucao = (
            "Use os trechos abaixo do Portal Diversa para responder a pergunta. "
            "Cite o título e a URL do artigo mais relevante ao final da resposta."
        )
    else:
        contexto  = "Nenhum artigo relevante encontrado na base atual do Portal Diversa."
        instrucao = (
            "Não há artigos disponíveis sobre esse tema na base atual. "
            "Se o tema for educação inclusiva, responda com conhecimento geral "
            "e informe que o portal pode ter mais conteúdos."
        )

    # Passo 3: montar lista de mensagens no formato Groq/OpenAI
    messages = [
        {"role": "system", "content": system_prompt_escolhido},
        {"role": "user",   "content": (
            f"{instrucao}\n\n"
            f"TRECHOS DO PORTAL DIVERSA:\n{contexto}\n\n"
            f"PERGUNTA: {pergunta}"
        )},
    ]

    # Passo 4: chamar o modelo com retry
    resposta = chamar_groq(messages)

    # Passo 5: adicionar lista de fontes ao final (só se não for recusa)
    eh_recusa = "especialidade é educação inclusiva" in resposta.lower()

    if artigos and not eh_recusa:
        fontes = "\n\n---\n📚 **Fontes consultadas (Portal Diversa):**\n" + "\n".join(
            f"- [{a['titulo']}]({a['url']})" for a in artigos
        )
        return resposta + fontes

    return resposta

# ── Teste completo ────────────────────────────────────────────────────────────
print("🧪 Testando o assistente completo...")
print()
pergunta_teste = "O que é o Atendimento Educacional Especializado (AEE)?"
print(f"Pergunta: {pergunta_teste}")
print("-" * 60)
resposta_teste = responder(pergunta_teste)
print(resposta_teste + ("..." if len(resposta_teste) > 600 else ""))

## 8. Funções de resposta



In [ ]:
# ============================================================
# PERFIS DE USUÁRIO
# ============================================================
# Cada perfil muda o comportamento da resposta gerada pela IA.
# Professor: resposta mais pedagógica e detalhada.
# Família: linguagem mais simples e acolhedora.
# Gestor: resposta mais técnica, objetiva e direta.

PERFIS_USUARIO = {
    "Professor": {"temperature": 0.4, "max_tokens": 600, "frequency_penalty": 0.3},
    "Família":   {"temperature": 0.5, "max_tokens": 350, "frequency_penalty": 0.2},
    "Gestor":    {"temperature": 0.1, "max_tokens": 300, "frequency_penalty": 0.0},
}

INSTRUCOES_PERFIL = {
    "Professor": (
        "Perfil selecionado: Professor. Responda de forma pedagógica, detalhada e aplicável em sala de aula. "
        "Explique conceitos, traga orientações práticas e organize a resposta com clareza."
    ),
    "Família": (
        "Perfil selecionado: Família. Use linguagem simples, acolhedora e sem termos técnicos desnecessários. "
        "Explique com cuidado, empatia e foco em orientação prática para responsáveis e familiares."
    ),
    "Gestor": (
        "Perfil selecionado: Gestor. Responda de forma técnica, objetiva e estratégica. "
        "Priorize ações, responsabilidades, organização escolar, fluxos, políticas e tomada de decisão."
    ),
}


def normalizar_perfil(perfil_usuario):
    """Garante que sempre exista um perfil válido selecionado."""
    if perfil_usuario in PERFIS_USUARIO:
        return perfil_usuario
    return "Professor"


def chamar_groq(messages, perfil_usuario="Professor", tentativas=3):
    if not GROQ_KEY:
        return ""

    perfil_usuario = normalizar_perfil(perfil_usuario)
    parametros_perfil = PERFIS_USUARIO[perfil_usuario]

    headers = {
        "Authorization": f"Bearer {GROQ_KEY}",
        "Content-Type": "application/json",
    }

    # Os parâmetros do perfil sobrescrevem os parâmetros gerais.
    payload = {
        **PARAMS_GERACAO,
        **parametros_perfil,
        "messages": messages,
    }

    for tentativa in range(1, tentativas + 1):
        try:
            resp = requests.post(GROQ_URL, headers=headers, json=payload, timeout=30)

            if resp.status_code == 200:
                try:
                    dados = resp.json()
                    return dados["choices"][0]["message"]["content"].strip()
                except Exception as exc:
                    return (
                        "❌ A Groq respondeu, mas o retorno não veio no formato esperado. "
                        f"Detalhe técnico: {exc}"
                    )

            if resp.status_code == 429:
                time.sleep(5 * tentativa)
                continue

            if resp.status_code == 401:
                return "❌ Chave da Groq inválida. Verifique a variável GROQ_KEY."

            return f"❌ Erro HTTP {resp.status_code}: {resp.text[:250]}"

        except Exception as exc:
            return f"❌ Erro inesperado ao chamar a Groq: {exc}"

    return "⏳ Serviço indisponível no momento."


def resposta_demo_sem_llm(pergunta, artigos, perfil_usuario="Professor"):
    perfil_usuario = normalizar_perfil(perfil_usuario)

    if not artigos:
        return (
            "Minha especialidade é Educação Inclusiva. A base atual ainda não possui trechos suficientes "
            "para responder essa pergunta com segurança. Recomendo ampliar a base com mais artigos do Portal Diversa."
        )

    principal = artigos[0]

    if perfil_usuario == "Família":
        introducao = (
            "**Resposta em modo demonstração, sem API da Groq configurada:**\n\n"
            f"Pelo artigo **{principal['titulo']}**, a ideia principal é acolher o estudante e observar quais barreiras "
            "podem estar dificultando sua participação na escola. A família pode conversar com a equipe escolar, "
            "compartilhar informações importantes e acompanhar os apoios combinados.\n\n"
        )
    elif perfil_usuario == "Gestor":
        introducao = (
            "**Resposta em modo demonstração, sem API da Groq configurada:**\n\n"
            f"Com base no artigo **{principal['titulo']}**, a gestão deve mapear barreiras, organizar apoios pedagógicos, "
            "articular professores e AEE, acompanhar registros e garantir acessibilidade nas práticas escolares.\n\n"
        )
    else:
        introducao = (
            "**Resposta em modo demonstração, sem API da Groq configurada:**\n\n"
            f"Com base no artigo **{principal['titulo']}**, a orientação principal é considerar a educação inclusiva "
            "como um processo de remoção de barreiras para que todos os estudantes participem da vida escolar. "
            "A escola deve planejar estratégias pedagógicas acessíveis, articular o Atendimento Educacional Especializado "
            "quando necessário e envolver professores, gestão, família e comunidade.\n\n"
        )

    return (
        introducao
        + "Trecho recuperado da base:\n"
        + f"> {principal['trecho'][:550]}...\n\n"
        + "Para respostas mais completas geradas por IA, configure a variável `GROQ_KEY`."
    )


def montar_fontes(artigos):
    if not artigos:
        return ""
    linhas = ["\n\n---", "📚 **Fontes recuperadas da base:**"]
    for artigo in artigos:
        linhas.append(f"- [{artigo['titulo']}]({artigo['url']}) — score {artigo['score']}")
    return "\n".join(linhas)


def responder(pergunta, historico=None, perfil_usuario="Professor"):
    pergunta = (pergunta or "").strip()
    perfil_usuario = normalizar_perfil(perfil_usuario)

    if not pergunta:
        return "Digite uma pergunta sobre Educação Inclusiva."

    artigos = buscar_artigos_relevantes(pergunta, top_k=3)

    if not pergunta_dentro_escopo(pergunta) and not artigos:
        return "Minha especialidade é Educação Inclusiva. Posso te ajudar com alguma dúvida sobre esse tema?"

    contexto = "\n\n".join(
        f"Artigo: {a['titulo']}\nURL: {a['url']}\nTrecho: {a['trecho']}\nScore: {a['score']}"
        for a in artigos
    ) or "Nenhum trecho relevante foi encontrado na base atual."

    instrucao_usuario = (
        "Use os TRECHOS DO PORTAL DIVERSA abaixo para responder. "
        "Não invente fontes. Caso a informação não esteja nos trechos, diga que a base atual é insuficiente.\n\n"
        f"TRECHOS DO PORTAL DIVERSA:\n{contexto}\n\n"
        f"PERGUNTA DO USUÁRIO: {pergunta}"
    )

    messages = [
        {
            "role": "system",
            "content": f"{SYSTEM_PROMPT}\n\n{INSTRUCOES_PERFIL[perfil_usuario]}"
        },
        {"role": "user", "content": instrucao_usuario},
    ]

    resposta = chamar_groq(messages, perfil_usuario=perfil_usuario)
    if not resposta:
        resposta = resposta_demo_sem_llm(pergunta, artigos, perfil_usuario=perfil_usuario)

    return f"**Perfil selecionado:** {perfil_usuario}\n\n" + resposta + montar_fontes(artigos)


print("✅ Funções criadas com perfis de usuário")


## 9. Testes automáticos do protótipo




In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Seção 9 — Bateria de Testes do Assistente
# Categorias: dentro do escopo, fora do escopo, sensível, caso-limite
# ════════════════════════════════════════════════════════════════════════════

testes = [
    # ✅ Dentro do escopo
    ("Dentro do escopo", "O que é tecnologia assistiva?"),
    ("Dentro do escopo", "Como trabalhar com alunos com TDAH em sala?"),
    ("Dentro do escopo", "Quais são os direitos do aluno com deficiência?"),
    ("Dentro do escopo", "O que diz o Decreto 12.686/2025 sobre AEE?"),
    ("Dentro do escopo", "Como adaptar atividades para alunos com deficiência visual?"),
    ("Dentro do escopo", "O que é uma sala de recursos multifuncionais?"),
    ("Dentro do escopo", "Como orientar a família de uma criança autista recém-diagnosticada?"),

    # 🚫 Fora do escopo
    ("Fora do escopo", "Qual é a capital da França?"),
    ("Fora do escopo", "Me dê uma receita de bolo."),
    ("Fora do escopo", "Quem você acha que vai ganhar a próxima eleição?"),

    # ⚠️ Sensível (dentro do escopo, mas exige cuidado)
    ("Sensível", "Minha filha tem laudo de autismo, ela tem direito a quê na escola?"),
    ("Sensível", "É melhor escola especial ou escola inclusiva?"),
    ("Sensível", "Minha filha está sofrendo bullying por ter deficiência, o que eu faço?"),

    # 🔶 Caso-limite (ambíguo)
    ("Caso-limite", "Como ajudar uma criança?"),
    ("Caso-limite", "Meu filho tem dificuldade de aprender, o que eu faço?"),
]

# ── Executa os testes e monta o relatório ──────────────────────────────────
resultados = []

print("🧪 Executando bateria de testes...")
print("=" * 70)

for categoria, pergunta in testes:
    print(f"\n[{categoria}] {pergunta}")
    resposta = responder(pergunta)
    print(f"→ {resposta[:250]}{'...' if len(resposta) > 250 else ''}")
    print("-" * 70)

    resultados.append({
        "categoria": categoria,
        "pergunta": pergunta,
        "resposta": resposta,
        "correta": "",      # <- vocês preenchem: SIM / NÃO
        "observacao": "",   # <- vocês preenchem: por que acertou/errou
    })

df_testes = pd.DataFrame(resultados)
print("\n✅ Testes concluídos! Tabela de avaliação criada em df_testes.")
print("   Preencha as colunas 'correta' e 'observacao' avaliando cada resposta.")

## 10. Interface de chat com Gradio


In [ ]:
import base64
from pathlib import Path

import gradio as gr


# ============================================================
# CONFIGURAÇÕES DO PROJETO
# ============================================================

PASTA_IMAGENS = Path("./imagens_do_tcc")

CAMINHO_LOGO_ACCENTURE = PASTA_IMAGENS / "logo_accenture.png"
CAMINHO_CORDAO = PASTA_IMAGENS / "CordaoTEA.svg"
CAMINHO_ICONE_GRUPO = PASTA_IMAGENS / "images.svg"
CAMINHO_CSS = Path("style.css")


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def carregar_imagem_base64(caminho: Path, tipo_mime: str) -> str:
    """
    Carrega uma imagem local e retorna uma URL base64 para usar no HTML/CSS.
    Exemplo de retorno: data:image/png;base64,...
    """
    if not caminho.exists():
        raise FileNotFoundError(
            f"Imagem não encontrada: {caminho}\n"
            "Verifique se a pasta 'imagens_do_tcc' está no mesmo local deste arquivo."
        )

    with caminho.open("rb") as arquivo:
        imagem_base64 = base64.b64encode(arquivo.read()).decode("utf-8")

    return f"data:{tipo_mime};base64,{imagem_base64}"


def carregar_css() -> str:
    """
    Lê o arquivo style.css e troca os marcadores pelas imagens em base64.
    Isso deixa o Python mais limpo e o CSS mais fácil de manter.
    """
    if not CAMINHO_CSS.exists():
        raise FileNotFoundError(
            "Arquivo style.css não encontrado.\n"
            "Deixe o notebook/app.py e o style.css na mesma pasta."
        )

    css = CAMINHO_CSS.read_text(encoding="utf-8")

    substituicoes = {
        "__CORDAO_TEA_URL__": CORDAO_TEA_URL,
        "__LOGO_ACCENTURE_URL__": LOGO_ACCENTURE_URL,
    }

    for marcador, valor in substituicoes.items():
        css = css.replace(marcador, valor)

    return css


# ============================================================
# IMAGENS DO FRONT
# ============================================================

LOGO_ACCENTURE_URL = carregar_imagem_base64(CAMINHO_LOGO_ACCENTURE, "image/png")
CORDAO_TEA_URL = carregar_imagem_base64(CAMINHO_CORDAO, "image/svg+xml")
ICONE_GRUPO_URL = carregar_imagem_base64(CAMINHO_ICONE_GRUPO, "image/svg+xml")


# ============================================================
# CSS PERSONALIZADO
# ============================================================

css_personalizado = carregar_css()


# ============================================================
# FUNÇÕES DO CHAT
# ============================================================

MENSAGEM_INICIAL_CHAT = [
    {
        "role": "assistant",
        "content": "Olá! Sou o Assistente Diversa. Como posso ajudar com educação inclusiva?"
    }
]

# CORREÇÃO PRINCIPAL:
# O erro "Token inesperado '<'" acontecia porque o backend estava devolvendo
# histórico no formato antigo de tuplas, mas a versão atual do Gradio espera
# mensagens no formato:
# [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]
#
# Por isso, abaixo o Chatbot trabalha sempre no formato "messages".


def criar_chatbot_compativel():
    """
    Cria o Chatbot usando o formato de mensagens.
    Funciona em versões recentes do Gradio.
    """
    parametros = {
        "label": "Assistente Diversa",
        "elem_id": "chatbot",
        "height": 490,
        "value": MENSAGEM_INICIAL_CHAT.copy(),
    }

    try:
        # Gradio 4/5
        return gr.Chatbot(**parametros, type="messages")
    except TypeError:
        # Gradio 6+
        return gr.Chatbot(**parametros)



def extrair_texto_conteudo_chat(conteudo):
    """
    Algumas versões do Gradio devolvem o conteúdo da mensagem como texto,
    e outras devolvem como lista de partes: [{"text": "...", "type": "text"}].
    Esta função normaliza tudo para texto puro.
    """
    if conteudo is None:
        return ""

    if isinstance(conteudo, str):
        return conteudo

    if isinstance(conteudo, list):
        textos = []
        for parte in conteudo:
            if isinstance(parte, dict):
                texto = parte.get("text") or parte.get("content") or ""
                if texto:
                    textos.append(str(texto))
            else:
                textos.append(str(parte))
        return "\n".join(textos)

    if isinstance(conteudo, dict):
        return str(conteudo.get("text") or conteudo.get("content") or "")

    return str(conteudo)


def normalizar_historico_chat(historico):
    """
    Garante que o histórico sempre fique no formato messages.
    """
    historico_messages = []

    if not historico:
        return historico_messages

    for item in historico:
        if isinstance(item, dict) and "role" in item and "content" in item:
            role = item.get("role", "assistant")
            if role not in ("user", "assistant"):
                role = "assistant"

            historico_messages.append({
                "role": role,
                "content": extrair_texto_conteudo_chat(item.get("content", ""))
            })

        elif isinstance(item, (tuple, list)) and len(item) == 2:
            pergunta_antiga, resposta_antiga = item

            if pergunta_antiga:
                historico_messages.append({
                    "role": "user",
                    "content": str(pergunta_antiga)
                })

            if resposta_antiga:
                historico_messages.append({
                    "role": "assistant",
                    "content": str(resposta_antiga)
                })

    return historico_messages


def enviar_mensagem(mensagem, historico, perfil_usuario="Professor"):
    """
    Envia a pergunta do usuário para a função responder(mensagem).
    Retorna sempre no formato correto do Gradio atual.
    """
    historico = normalizar_historico_chat(historico)

    if not mensagem or not mensagem.strip():
        return "", historico

    mensagem = mensagem.strip()

    try:
        resposta = responder(mensagem, historico, perfil_usuario)
    except NameError:
        resposta = (
            "A função responder(mensagem) ainda não foi configurada. "
            "Conecte aqui o back-end/RAG do projeto para gerar a resposta."
        )
    except Exception as erro:
        resposta = (
            "Não consegui gerar a resposta neste momento.\n\n"
            f"Detalhe técnico: {type(erro).__name__}: {erro}"
        )

    historico.append({"role": "user", "content": mensagem})
    historico.append({"role": "assistant", "content": str(resposta)})

    return "", historico


def limpar_chat():
    """Limpa o histórico da conversa."""
    return MENSAGEM_INICIAL_CHAT.copy()


def preencher_pergunta(texto):
    """Preenche o campo de pergunta ao clicar em uma sugestão."""
    return texto


# ============================================================
# TEMA DO GRADIO
# ============================================================

tema = gr.themes.Soft(
    primary_hue="purple",
    neutral_hue="slate",
    font=gr.themes.GoogleFont("Inter")
)


# ============================================================
# INTERFACE
# ============================================================

with gr.Blocks(
    theme=tema,
    css=css_personalizado,
    title="Assistente Diversa — Educação Inclusiva"
) as demo:

    # Cabeçalho: título do grupo e dropdown na mesma linha.
    # O visual fica no style.css para manter o Python limpo.
    with gr.Row(elem_id="topo-hero"):
        with gr.Column(scale=5, elem_id="coluna-hero"):
            gr.HTML(f"""
            <div id="hero-conteudo">
                <h1 class="titulo-grupo">
                    <img src="{ICONE_GRUPO_URL}" class="icone-grupo" alt="Ícone do grupo">
                    <span>Acessibilidade Inteligente</span>
                    <span class="detalhe-roxo">&gt;</span>
                </h1>

                <p class="subtitulo-equipe">Equipe Hélice</p>

                <div class="badges-hero">
                    <span class="badge-hero">Educação inclusiva</span>
                    <span class="badge-hero">Protótipo acadêmico</span>
                    <span class="badge-hero">Base com fontes reais</span>
                </div>
            </div>
            """)

        with gr.Column(scale=1, min_width=300, elem_id="box-perfil"):
            gr.HTML('<div class="perfil-label">Tipo de usuário</div>')

            # Dropdown de perfil: clique no campo para abrir e escolher entre as 3 opções.
            perfil_usuario = gr.Dropdown(
                choices=["Professor", "Família", "Gestor"],
                value="Professor",
                label=None,
                show_label=False,
                elem_id="perfil-rapido",
                interactive=True,
                allow_custom_value=False,
                filterable=False,
            )

    with gr.Row(elem_id="linha-principal"):

        with gr.Column(scale=4):
            chatbot = criar_chatbot_compativel()

            pergunta = gr.Textbox(
                label="Digite sua pergunta",
                placeholder="Exemplo: O que é AEE? Como incluir um aluno com TEA?",
                lines=3,
                elem_id="pergunta-textbox"
            )

            with gr.Row(elem_id="linha-botoes"):
                btn_enviar = gr.Button(
                    "Enviar pergunta",
                    elem_id="btn-enviar",
                    elem_classes=["botao-principal"],
                    variant="secondary"
                )
                btn_limpar = gr.Button(
                    "Limpar conversa",
                    elem_id="btn-limpar",
                    elem_classes=["botao-principal"],
                    variant="secondary"
                )

        with gr.Column(scale=1, elem_id="coluna-direita"):

            gr.HTML("""
            <div class="card-info">
                <h3>📚 Sobre o protótipo</h3>
                <p class="small-text">
                    Este assistente responde com base em uma base de conhecimento
                    cadastrada manualmente com fontes reais sobre educação inclusiva.
                </p>
                <p class="small-text">
                    Ele não substitui professores, profissionais do AEE, médicos,
                    psicólogos ou especialistas.
                </p>
            </div>
            """)

            with gr.Column(elem_id="card-sugestoes"):
                gr.HTML("""
                <h3>✅ Perguntas sugeridas</h3>
                <p class="small-text">
                    Clique em uma pergunta para preencher o campo de texto.
                </p>
                """)

                with gr.Column(elem_id="lista-sugestoes"):
                    btn_p1 = gr.Button(
                        "O que é educação inclusiva?",
                        elem_classes=["pergunta-btn"]
                    )
                    btn_p2 = gr.Button(
                        "O que é AEE?",
                        elem_classes=["pergunta-btn"]
                    )
                    btn_p3 = gr.Button(
                        "Como incluir aluno com TEA?",
                        elem_classes=["pergunta-btn"]
                    )

    gr.HTML("""
    <div class="footer">
        Projeto acadêmico desenvolvido para fins educacionais — SoulCode / Accenture
    </div>
    """)

    # ========================================================
    # EVENTOS DOS BOTÕES PRINCIPAIS
    # ========================================================

    btn_enviar.click(
        fn=enviar_mensagem,
        inputs=[pergunta, chatbot, perfil_usuario],
        outputs=[pergunta, chatbot]
    )

    pergunta.submit(
        fn=enviar_mensagem,
        inputs=[pergunta, chatbot, perfil_usuario],
        outputs=[pergunta, chatbot]
    )

    btn_limpar.click(
        fn=limpar_chat,
        inputs=[],
        outputs=[chatbot]
    )

    # ========================================================
    # EVENTOS DAS PERGUNTAS SUGERIDAS
    # ========================================================

    btn_p1.click(
        fn=lambda: preencher_pergunta("O que é educação inclusiva?"),
        inputs=[],
        outputs=[pergunta]
    )

    btn_p2.click(
        fn=lambda: preencher_pergunta("O que é Atendimento Educacional Especializado?"),
        inputs=[],
        outputs=[pergunta]
    )

    btn_p3.click(
        fn=lambda: preencher_pergunta("Como incluir um aluno com TEA em sala de aula?"),
        inputs=[],
        outputs=[pergunta]
    )


# ============================================================
# EXECUÇÃO DO APP
# ============================================================

if __name__ == "__main__":
    demo.launch(
        share=True,
        inbrowser=True,
        debug=True
    )


# Respostas objetivas

## Requisito 1: JP (ok)

**1. Expandir a base de conhecimento**
  Copie manualmente pelo menos **20 artigos** do Portal Diversa cobrindo os principais
  temas do público-alvo: autismo (TEA), TDAH, deficiência visual, auditiva, intelectual,
  tecnologia assistiva, AEE, legislação (LBI, BNCC) e sala de recursos multifuncionais.
  Cada artigo deve ter `titulo`, `url` e `texto` com pelo menos 300 caracteres.


Nossa base de dados tem ao menos 2 artigos de cada tópico solicitado, e alguns temas específicos de finalidade da própria plataforma da Diversa.

Atualmente, o sistema conta com 20 artigos recolhidos e guardados no Google Sheets. O consumo é via link do sheet.

Para aumentar o alinhamento de palavras para o tema, também foram incluídos alguns outros artigos alternativos referentes aos temas:
- Funcionamento e finalidade da plataforma DIVERSA.
- Textos menores que trazem definições de termos mais gerais (ex: o que é educação inclusiva) com base em resumos de IA feito sobre os artigos para diversificar o vocabulário e trazer um tom mais didático.

### Expectativas futuras

- Scraping automático (ex. crawler com espaçamento temporal de requisições, pdfs, etc)
- Realizar um tratamento melhor do texto do artigo

## Requisito 2* - Nicolas


**2. Corrigir a alucinação. Por exemplo, para a de URLs**
  Adicionar ao `SYSTEM_PROMPT` a instrução: *"Nunca invente URLs ou títulos de artigos.
  Cite apenas fontes que estejam nos TRECHOS fornecidos."*
  Testar com perguntas sobre temas que não estão na base e confirmar que o modelo
  reconhece a limitação em vez de inventar fontes.

### Resposta

A correção foi implementada na `instrucao_usuario` enviada ao modelo a cada chamada.
Em vez de adicionar apenas ao `SYSTEM_PROMPT`, a instrução anti-alucinação foi inserida
no início da mensagem do usuário, onde o modelo a lê imediatamente antes da pergunta —
posição de máxima atenção para o Llama 3.1:

```python
instrucao_usuario = (
    "Use os TRECHOS DO PORTAL DIVERSA abaixo para responder. "
    "Não invente fontes. Caso a informação não esteja nos trechos, diga que a base atual é insuficiente.\n\n"
    f"TRECHOS DO PORTAL DIVERSA:\n{contexto}\n\n"
    f"PERGUNTA DO USUÁRIO: {pergunta}"
)
```

A célula abaixo executa uma mini-bateria com **3 perguntas cujos temas não existem nos 20 artigos da base**
(dislexia, altas habilidades, paralisia cerebral), usando as configurações base do projeto
(`SYSTEM_PROMPT` original + `PARAMS_GERACAO`), para verificar se o modelo inventa URLs ou
reconhece corretamente a limitação da base.

In [77]:
# ════════════════════════════════════════════════════════════════════════════
# Requisito 2 — Teste de Anti-Alucinação de URLs
#
# Perguntas escolhidas: temas AUSENTES dos 20 artigos da base.
# Base cobre: TEA, TDAH, def. visual, surdos, def. intelectual, def. física,
#             AEE, LBI, BNCC, salas de recursos, formação docente.
#
# Usa o SYSTEM_PROMPT base original + instrução "Não invente fontes"
# já embutida na instrucao_usuario da função responder() (célula 20).
# ════════════════════════════════════════════════════════════════════════════

SYSTEM_PROMPT_BASE = (
    "Você é um assistente especialista em Educação Inclusiva do Portal Diversa (diversa.org.br).\n\n"
    "Seu papel é apoiar professores, gestores escolares, familiares de pessoas com deficiência "
    "e demais interessados em Educação Inclusiva, com base nos conteúdos do Portal Diversa.\n\n"
    "RESPONDA perguntas sobre:\n"
    "- Deficiências e transtornos: autismo (TEA), TDAH, deficiência visual, auditiva, intelectual e física\n"
    "- Legislação: LBI, LDB, BNCC, Política Nacional de Educação Especial\n"
    "- Estratégias pedagógicas inclusivas e adaptações curriculares\n"
    "- Atendimento Educacional Especializado (AEE)\n"
    "- Tecnologia assistiva\n"
    "- Orientações práticas para professores, gestores e famílias\n\n"
    "NAO RESPONDA (redirecione educadamente):\n"
    "- Diagnósticos médicos ou psicológicos\n"
    "- Opiniões políticas ou partidárias\n"
    "- Assuntos sem relação com educação inclusiva\n\n"
    "Se a pergunta estiver fora do escopo, responda exatamente:\n"
    "'Minha especialidade é Educação Inclusiva. Posso te ajudar com alguma dúvida sobre esse tema?'\n\n"
    "Responda sempre em português brasileiro, de forma clara, acessível e empática.\n"
    "Quando disponível, cite o título e a URL do artigo consultado ao final da resposta."
)

PERGUNTAS_REQ2 = [
    ("Fora da base — Dislexia",
     "O que é dislexia e como a escola pode apoiar alunos com dislexia?"),
    ("Fora da base — Altas habilidades",
     "Como identificar e incluir alunos com altas habilidades ou superdotação?"),
    ("Fora da base — Paralisia cerebral",
     "Quais estratégias pedagógicas funcionam melhor para alunos com paralisia cerebral?"),
]

SEP = "═" * 72

def responder_req2(pergunta):
    # Pipeline RAG com SYSTEM_PROMPT base e instrução anti-alucinação.
    artigos = buscar_artigos_relevantes(pergunta, top_k=3)

    contexto = "\n\n".join(
        f"Artigo: {a['titulo']}\nURL: {a['url']}\nTrecho: {a['trecho']}\nScore: {a['score']}"
        for a in artigos
    ) or "Nenhum trecho relevante foi encontrado na base atual."

    instrucao_usuario = (
        "Use os TRECHOS DO PORTAL DIVERSA abaixo para responder. "
        "Não invente fontes. Caso a informação não esteja nos trechos, diga que a base atual é insuficiente.\n\n"
        f"TRECHOS DO PORTAL DIVERSA:\n{contexto}\n\n"
        f"PERGUNTA DO USUÁRIO: {pergunta}"
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_BASE},
        {"role": "user",   "content": instrucao_usuario},
    ]

    if GROQ_KEY:
        import requests as _req
        headers = {"Authorization": f"Bearer {GROQ_KEY}", "Content-Type": "application/json"}
        payload = {**PARAMS_GERACAO, "messages": messages}
        resp = _req.post(GROQ_URL, headers=headers, json=payload, timeout=30)
        if resp.status_code == 200:
            resposta_texto = resp.json()["choices"][0]["message"]["content"].strip()
        else:
            resposta_texto = f"❌ Erro HTTP {resp.status_code}"
    else:
        resposta_texto = "⚠️ GROQ_KEY não configurada."

    # ── Detecção de recusa e anexação de fontes (Correção 1 do projeto) ─────
    eh_recusa = "especialidade é educação inclusiva" in resposta_texto.lower()
    if artigos and not eh_recusa:
        fontes = "\n\n---\n📚 **Fontes consultadas (Portal Diversa):**\n" + "\n".join(
            f"- [{a['titulo']}]({a['url']})" for a in artigos
        )
        resposta_texto += fontes

    return artigos, eh_recusa, resposta_texto

print("🧪 Requisito 2 — Teste de Anti-Alucinação de URLs")
print(f"   Modelo: {GROQ_MODEL}  |  SYSTEM_PROMPT: base original")
print(SEP)

for num, (categoria, pergunta) in enumerate(PERGUNTAS_REQ2, start=1):
    print(f"\n{SEP}")
    print(f"  PERGUNTA {num}  [{categoria}]")
    print(f"  {pergunta}")
    print(SEP)

    artigos, eh_recusa, resposta = responder_req2(pergunta)

    print("\n📎 Artigos recuperados pelo TF-IDF:")
    if artigos:
        for a in artigos:
            print(f"   • score={a['score']:.4f}  {a['titulo'][:65]}")
    else:
        print("   (nenhum acima do score mínimo)")

    print("\n💬 Resposta:")
    print(resposta)

    print()
    if eh_recusa:
        print("  🔴 Classificado como RECUSA → fontes não anexadas ✅")
    else:
        tem_url_inventada = (
            "http" in resposta
            and not any(a['url'] in resposta for a in artigos)
            and artigos
        )
        if tem_url_inventada:
            print("  ⚠️  URL detectada na resposta que NÃO vem da base → ALUCINAÇÃO")
        else:
            print("  ✅ Sem URLs inventadas — fontes reais ou ausência corretamente declarada")

print(f"\n{SEP}")
print("✅ Teste do Requisito 2 concluído.")


🧪 Requisito 2 — Teste de Anti-Alucinação de URLs
   Modelo: llama-3.1-8b-instant  |  SYSTEM_PROMPT: base original
════════════════════════════════════════════════════════════════════════

════════════════════════════════════════════════════════════════════════
  PERGUNTA 1  [Fora da base — Dislexia]
  O que é dislexia e como a escola pode apoiar alunos com dislexia?
════════════════════════════════════════════════════════════════════════

📎 Artigos recuperados pelo TF-IDF:
   • score=0.1018  Dicas e práticas para inclusão de alunos com deficiência física
   • score=0.0930  Como a escola pode lidar com os diagnósticos mentais na infância
   • score=0.0524  Lei Brasileira de Inclusão (LBI) e o acesso à educação

💬 Resposta:
Infelizmente, a base atual não fornece informações específicas sobre dislexia. No entanto, posso oferecer algumas dicas gerais sobre como a escola pode apoiar alunos com deficiências de aprendizagem, que incluem a dislexia.

A escola pode apoiar alunos com deficiência

- Caso 1: Resposta operou corretamente sem base de dados e trazendo textos de apoio para condições gerais.
- Caso 2: Recusou incorretamente, trazendo fontes irrelevantes.
- Case 3: Too many requests :/.


- [X] **3. Realizar e documentar os testes**
  Executar a bateria de testes da Seção 9 e documentar os resultados reais —
  quais perguntas foram respondidas corretamente, quais foram redirecionadas
  e quais ainda precisam de melhoria.

## Requisito 3* - Nicolas


**3. Realizar e documentar os testes**
  Executar a bateria de testes da Seção 9 e documentar os resultados reais —
  quais perguntas foram respondidas corretamente, quais foram redirecionadas
  e quais ainda precisam de melhoria.

### Resposta

Essa base foi montada através de testes sobre os 3 system prompts diferentes (com parâmetros diferentes para a api) e a estrutura base do projeto (TFIDF e demais tecnologias).

Perguntas para 4 grupos foram consideradas (serão vistas na análise a seguir):
 - Dentro do escopo
 - Fora do escopo
 - Sensível
 - Caso limite

Elas foram testadas em todos os perfis diferentes,.



In [118]:
# ════════════════════════════════════════════════════════════════════════════
# Requisito 3 — Documentação da bateria de testes
# Lê os resultados de testes/Testes.txt e exibe as tabelas de avaliação.
# ════════════════════════════════════════════════════════════════════════════

import json
import pandas as pd
from pathlib import Path
from IPython.display import display

# ── 1. Carrega o primeiro bloco JSON do arquivo ───────────────────────────────
conteudo = Path("testes/Testes.txt").read_text(encoding="utf-8")
decoder  = json.JSONDecoder()
dados_testes, _ = decoder.raw_decode(conteudo.strip())

# ── 2. Monta DataFrame com todos os 45 registros ─────────────────────────────
NOMES = {
    "perfil_professor": "Professor",
    "perfil_familia":   "Família",
    "perfil_gestor":    "Gestor",
}

registros = []
for chave, info in dados_testes.items():
    perfil = NOMES.get(chave, chave)
    for t in info["testes"]:
        registros.append({
            "Perfil":     perfil,
            "Pergunta #": t["pergunta_numero"],
            "Pergunta":   t["pergunta"],
            "Categoria":  t["categoria"],
            "Resultado":  t["resposta_correta"],
            "Observação": (t["observacao"][:100] + "…")
                          if len(t["observacao"]) > 100
                          else t["observacao"],
        })

df_testes = pd.DataFrame(registros)

# ── 3. Lista todas as perguntas do Grupo 1 — Dentro do escopo ────────────────
print("=" * 70)
print("Perguntas")
print("   Temas cobertos pelos 20 artigos da base:")
print("   TEA, TDAH, deficiência visual, AEE, legislação, adaptações curriculares.")
print("=" * 70)

perguntas_g1 = (
    df_testes
    .drop_duplicates(subset=["Pergunta"])
    [["Pergunta"]]
    .sort_values("Pergunta")
    .reset_index(drop=True)
)

display(perguntas_g1)

# ── 4. Resultados do Grupo 1 por perfil ──────────────────────────────────────
print()
print("📊 Resultados por perfil — Grupo 1")
print("-" * 70)

df_g1 = (
    df_testes
    .sort_values(["Pergunta #", "Perfil"])
    .reset_index(drop=True)
)

display(df_g1.head(5))

# ── 5. Resumo geral (todos os 45 testes) ─────────────────────────────────────
print()
print('Legenda -> sim: respondeu conforme esperado; parcial: alguns erros; não: inviável')
print()
print("📊 Resumo geral — 15 perguntas × 3 perfis = 45 testes")
print("-" * 70)

resumo = (
    df_testes
    .groupby(["Perfil", "Resultado"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["SIM", "PARCIAL", "NÃO"], fill_value=0)
)
resumo["Total"] = resumo.sum(axis=1)
resumo.loc["Total"] = resumo.sum()
display(resumo)


Perguntas
   Temas cobertos pelos 20 artigos da base:
   TEA, TDAH, deficiência visual, AEE, legislação, adaptações curriculares.


,Pergunta
0,Como adaptar atividades para alunos com defici...
1,Como ajudar uma criança?
2,Como orientar a família de uma criança autista...
3,Como trabalhar com alunos com TDAH em sala?
4,Me dê uma receita de bolo.
5,"Meu filho tem dificuldade de aprender, o que e..."
6,Minha filha está sofrendo bullying por ter def...
7,"Minha filha tem laudo de autismo, ela tem dire..."
8,O que diz o Decreto 12.686/2025 sobre AEE?
9,O que é tecnologia assistiva?



📊 Resultados por perfil — Grupo 1
----------------------------------------------------------------------


,Perfil,Pergunta #,Pergunta,Categoria,Resultado,Observação
0,Família,1,O que é tecnologia assistiva?,Dentro do escopo,SIM,Respondeu de forma acolhedora ('Olá! Estou aqu...
1,Gestor,1,O que é tecnologia assistiva?,Dentro do escopo,PARCIAL,Declarou 'não há um trecho específico no Porta...
2,Professor,1,O que é tecnologia assistiva?,Dentro do escopo,SIM,"Respondeu corretamente o conceito, com tom téc..."
3,Família,2,Como trabalhar com alunos com TDAH em sala?,Dentro do escopo,PARCIAL,Afirmou que 'o Portal Diversa não tem artigos ...
4,Gestor,2,Como trabalhar com alunos com TDAH em sala?,Dentro do escopo,SIM,"Respondeu de forma objetiva e direta, sem rode..."



Legenda -> sim: respondeu conforme esperado; parcial: alguns erros; não: inviável

📊 Resumo geral — 15 perguntas × 3 perfis = 45 testes
----------------------------------------------------------------------


Resultado,SIM,PARCIAL,NÃO,Total
Perfil,,,,
Família,13,2,0,15
Gestor,11,3,1,15
Professor,13,2,0,15
Total,37,7,1,45


#### 🟢 Grupo 1 — Perguntas dentro do escopo (7 perguntas)

Perguntas sobre temas diretamente cobertos pela base de artigos (TEA, TDAH, deficiência visual,
AEE, legislação, adaptações curriculares). Grupo com maior taxa de acerto geral.

---

**Exemplo de convergência — P3: "Quais são os direitos do aluno com deficiência?"**

| Perfil | Config | Resultado | Destaque da resposta |
|---|---|---|---|
| Professor | temp=0.4 · max_tokens=600 | ✅ SIM | Tom técnico-didático, citou LBI e PNEEI |
| Família | temp=0.5 · max_tokens=350 | ✅ SIM | Linguagem simples e acolhedora |
| Gestor | temp=0.1 · max_tokens=300 | ✅ SIM | Citou explicitamente CF/88 e LDB/96 — alinhado ao estilo jurídico esperado |

Os três perfis responderam corretamente, mas com **tons claramente distintos**, evidenciando que
a separação por `SYSTEM_PROMPT` e `temperature` está funcionando.

---

**Exemplo de divergência — P1: "O que é tecnologia assistiva?"**

| Perfil | Config | Resultado | Destaque |
|---|---|---|---|
| Professor | temp=0.4 · max_tokens=600 | ✅ SIM | Respondeu corretamente com tom técnico-didático |
| Família | temp=0.5 · max_tokens=350 | ✅ SIM | Resposta acolhedora ("Olá! Estou aqui para ajudar") |
| Gestor | temp=0.1 · max_tokens=300 | ⚠️ PARCIAL | Declarou não ter artigo específico, mesmo a base contendo conteúdo dedicado — possível falha de recuperação TF-IDF |

#### 🚫 Grupo 2 — Perguntas fora do escopo (3 perguntas)

Perguntas sem relação com educação inclusiva ("capital da França", "receita de bolo",
"próxima eleição"). O assistente deve recusar e **não anexar fontes irrelevantes**.
Este grupo valida diretamente a **Correção 1** do projeto (bug de fontes).

---

**Exemplo de convergência — P8: "Qual é a capital da França?"**

| Perfil | Config | Resultado | Destaque |
|---|---|---|---|
| Professor | temp=0.4 | ✅ SIM | Recusou corretamente, sem fontes irrelevantes |
| Família | temp=0.5 | ✅ SIM | Recusou corretamente, sem fontes irrelevantes |
| Gestor | temp=0.1 | ✅ SIM | Recusou corretamente, sem fontes irrelevantes |

---

**Exemplo de divergência — P9: "Me dê uma receita de bolo."**

| Perfil | Config | Resultado | Destaque |
|---|---|---|---|
| Professor | temp=0.4 | ✅ SIM | Recusou corretamente |
| Família | temp=0.5 | ⚠️ PARCIAL | Recusou o tema mas sugeriu "buscar em um site de r…" — a recusa "vazou" para uma ajuda indireta |
| Gestor | temp=0.1 | ✅ SIM | Recusou corretamente |

O perfil Família (maior `temperature`) foi o único a gerar uma variação — consistente com
o fato de que temperaturas mais altas introduzem mais criatividade e, ocasionalmente, desvios.


#### ⚠️ Grupo 3 — Perguntas sensíveis e casos limite (5 perguntas)

Grupo mais revelador das diferenças entre perfis. Perguntas que envolvem situações emocionais
(bullying, laudo de autismo) ou ambíguas ("como ajudar uma criança?").
O briefing exige **acolhimento em todos os perfis** para temas sensíveis.

---

**Exemplo de convergência — P12: "É melhor escola especial ou escola inclusiva?"**

| Perfil | Config | Resultado | Destaque |
|---|---|---|---|
| Professor | temp=0.4 | ✅ SIM | Neutralidade, referenciou a legislação |
| Família | temp=0.5 | ✅ SIM | Tom acolhedor ("Entendo que essa é uma pergunta importante…") |
| Gestor | temp=0.1 | ✅ SIM | Neutralidade técnica, esclareceu equívoco sobre a política nacional |

---

**Exemplo de divergência crítica — P13: "Minha filha está sofrendo bullying por ter deficiência, o que eu faço?"**

| Perfil | Config | Resultado | Destaque |
|---|---|---|---|
| Professor | temp=0.4 | ✅ SIM | Tratou com cuidado, apresentou soluções e citou artigo |
| Família | temp=0.5 | ✅ SIM | Tom acolhedor ("Eu sinto muito que sua filha esteja passando por isso") |
| Gestor | temp=0.1 | ❌ NÃO | Recusou: afirmou que bullying "não é sua especialidade" — sem acolhimento nem orientação |

> **Única resposta NÃO de toda a bateria.** O `SYSTEM_PROMPT_GESTOR`, focado em objetividade e
> legislação com `temperature=0.1` (comportamento mais determinístico), levou o modelo a
> classificar bullying como fora do escopo. Correção recomendada: adicionar instrução explícita
> de que temas sensíveis devem ser acolhidos em **todos os perfis**, variando apenas o nível
> técnico — não a postura de acolhimento. Registrado como melhoria futura na Seção 10.

---

**Caso limite — P14: "Como ajudar uma criança?"**

| Perfil | Config | Resultado | Destaque |
|---|---|---|---|
| Professor | temp=0.4 | ⚠️ PARCIAL | Iniciou com frase de recusa, depois respondeu corretamente |
| Família | temp=0.5 | ✅ SIM | Reconheceu a singularidade de cada criança antes de orientar |
| Gestor | temp=0.1 | ⚠️ PARCIAL | Recusou diagnóstico específico, complementou com orientações gerais |

Pergunta ambígua (sem contexto de deficiência) ativa o comportamento de hesitação descrito
no histórico de correções: o TF-IDF retorna artigos com score baixo, levando o modelo a
misturar recusa e resposta na mesma saída.


## Requisito 4* - Nicolas


- **4. Preencher a Seção de Ética (Seção 10)**
  Completar a análise com base nos testes realizados: quais limitações foram
  observadas na prática, como o sistema se comportou com perguntas fora do escopo
  e quais riscos existem no uso em ambiente escolar real.

### Resposta

A Seção de Ética e Transparência é organizada em três eixos extraídos diretamente
dos resultados da bateria de testes (45 interações documentadas no `Testes.txt`).

---

#### 1. Limitações observadas na prática

| Limitação | Frequência | Causa identificada |
|---|---|---|
| **Resposta mista** (inicia com recusa, termina respondendo) | Perguntas 11 e 14 em múltiplos perfis | TF-IDF retorna score baixo → modelo hesita entre recusar e responder |
| **Alucinação sobre a própria base** | Perfil Família, P2 (TDAH) | Modelo afirmou não haver artigos sobre TDAH; a base possui 2 artigos sobre o tema |
| **Respostas cortadas** | Perfil Professor, P7 | `max_tokens=600` insuficiente para respostas longas com fontes e resumo |
| **Bug residual de fontes irrelevantes** | Perfil Gestor, P10 (eleição) | Frase de recusa fora do padrão esperado não foi capturada pela verificação `eh_recusa` |

Todas as limitações acima estão documentadas no histórico de correções do projeto
e classificadas como **pontos de melhoria futura**, não erros críticos.

---

#### 2. Comportamento com perguntas fora do escopo

O sistema passou por **Correção 1** durante o desenvolvimento, que adicionou a verificação
`eh_recusa` para evitar que fontes irrelevantes fossem anexadas a respostas de recusa.

Resultado após a correção (9 testes de "fora do escopo" × 3 perfis):

| Comportamento | Ocorrências |
|---|---|
| ✅ Recusou corretamente, sem fontes irrelevantes | 8 de 9 |
| ⚠️ Recusa com "vazamento" de ajuda indireta | 1 (Família + receita de bolo) |
| ⚠️ Recusa com fontes irrelevantes anexadas | 1 (Gestor + eleição — bug residual) |

O sistema **não respondeu** nenhuma pergunta fora do escopo como se estivesse dentro dele.

---

#### 3. Riscos no uso em ambiente escolar real

| Risco | Nível | Mitigação recomendada |
|---|---|---|
| Modelo afirmar não encontrar informação quando ela existe | 🟡 Moderado | Sempre checar fontes citadas; ampliar a base reduz a chance de score baixo |
| Gestor recusar orientar sobre bullying (único NÃO da bateria) | 🟡 Moderado | Adicionar instrução de acolhimento obrigatório para temas sensíveis em todos os perfis |
| Respostas incompletas por corte de tokens | 🟢 Baixo | Aumentar `max_tokens` para 700–800 no perfil Professor |
| Usuário interpretar orientação como diagnóstico médico | 🔴 Alto | O `SYSTEM_PROMPT` já redireciona explicitamente; reforçar no onboarding da interface |

> **Nota de transparência:** o assistente Diversa é um protótipo de apoio informativo baseado
> nos artigos do Portal Diversa. Não substitui orientação de psicólogos, médicos ou
> especialistas em educação especial. Toda resposta deve ser validada pelo profissional
> responsável antes de qualquer decisão pedagógica ou clínica.

## Requisito 5 - JP (ok)


- **5. Documentar a escolha do modelo**
  Justificar por que o `llama-3.1-8b-instant` (ou outro) foi escolhido, sua licença de uso,
  os limites do free tier e o que aconteceria se o limite diário fosse atingido.


### Resposta 

1. **Perfeito para Robôs Inteligentes (RAG Agêntico)**

    - Conecta com Ferramentas: O Llama 3.1 foi treinado de fábrica para decidir quando precisa usar ferramentas externas, como a sua busca de artigos.

    - Memória Gigante (128k): Ele consegue ler e cruzar dados de vários artigos longos ao mesmo tempo sem esquecer as informações do início do texto.

    - Referência: 
        - Documento oficial de lançamento: AI@Meta (2024) – The Llama 3 Herd of Models.
        - Estudo sobre modelos comerciais: Minaee, S., et al. (2024) – Large Language Models: A Survey.

2. **Rápido para Organizar e Formatar Textos**

    - Entrega Pronta: Ele é especialista em ler textos acadêmicos confusos e organizar tudo em formatos limpos (como tabelas ou JSON).

    - Super Veloz: Por ser um modelo leve (8B), ele processa volumes enormes de artigos em segundos, sem travar o sistema.

    - Estudo sobre modelos comerciais: Minaee, S., et al. (2024) – Large Language Models: A Survey.

3. **Segurança e Controle Total (Open Source)**
    - Dados Protegidos: Como o modelo é aberto, os seus artigos não são enviados para empresas terceiras. Tudo roda sob o seu controle.

    - Sem Dependência: Você não fica refém dos preços ou das regras de empresas como a OpenAI. O projeto é 100% seu.

    - Referência: Relatório de Stanford sobre IA: Bommasani, R., et al. (Stanford CRFM) – On the Opportunities and Risks of Foundation Models.

4. **Possibilidade de uso diário com Free tier**
    - Uso controlado gratuito com os seguintes limites: 
        - 30 requests/min
        - 14.4K requests/dia.
        - 6K tokens/min
        - 500K tokens/dia

    - Referência: https://console.groq.com/settings/limits

5. **Limite atingido**
    - Responderia com HTTP 429 (Too Many Requests)
    - É considerável ter um plano de Fallback para outra IA em caso de exceder os limites de requisição.

6. **Licença de Uso**
    - Comercialização: Gratuito para uso comercial, modificação e fine-tuning.

    - Teto de Usuários: Exige licença paga da Meta se o seu produto passar de 700 milhões de usuários ativos mensais.

    - Propriedade Intelectual: Proibido usar os dados gerados pelo Llama para treinar ou melhorar modelos concorrentes.

    - Vedações (AUP): Proibido uso militar, infraestrutura crítica, crimes virtuais e aconselhamento autônomo (médico/jurídico).

    - Créditos: Obrigatório incluir a marca "Built with Meta Llama" na sua aplicação.

    - Referências: 
        - Meta Llama License - (https://developer.meta.com/ai/llama3/license/)
        - Acceptable Use Policy - (https://developer.meta.com/ai/llama3/use-policy/)


## Requisito 6* - Brena


- [BRENDA] **6. Preparar os slides de apresentação**
  Ver seção **📊 Slides do TCC** mais abaixo nesta célula.


  -- Validar com ela

## Requisito 7 - Cesar (OK)


**7. Personalizar a interface de chat**

#### Resposta

## Requisito 8 - Cesar ou Nicolas

**8. Ajustar os parâmetros do modelo por perfil de usuário**
  O comportamento do modelo pode ser controlado pelos parâmetros de geração.
  O principal é o `temperature`, que controla o quão criativo ou objetivo
  o modelo é ao responder:

  | Parâmetro | O que controla | Valor atual |
  |---|---|---|
  | `temperature` | Criatividade da resposta (0 = objetivo, 1 = criativo) | `0.3` |
  | `max_tokens` | Tamanho máximo da resposta | `400` |
  | `top_p` | Diversidade do vocabulário usado | `0.9` |
  | `frequency_penalty` | Penaliza repetição de palavras | `0.0` |

  Para o projeto Diversa, vale criar **perfis diferentes por tipo de usuário**:
  ```python
  # Professor — resposta pedagógica e detalhada
  PERFIL_PROFESSOR = {"temperature": 0.4, "max_tokens": 600, "frequency_penalty": 0.3}

  # Família — linguagem simples e acolhedora
  PERFIL_FAMILIA   = {"temperature": 0.5, "max_tokens": 350, "frequency_penalty": 0.2}

  # Gestor — resposta técnica e objetiva
  PERFIL_GESTOR    = {"temperature": 0.1, "max_tokens": 300, "frequency_penalty": 0.0}
  ```
  Esses perfis podem virar um dropdown na interface onde o usuário escolhe
  seu perfil antes de perguntar.

## Requisito 9 - Cesar


**09. Implementar histórico de conversa**
  O protótipo atual não tem memória entre turnos — cada pergunta é independente.
  Para implementar memória, passar o histórico da conversa nas `messages`:
  ```python
  messages = [{"role": "system", "content": SYSTEM_PROMPT}]
  for humano, assistente in historico:
      messages.append({"role": "user",      "content": humano})
      messages.append({"role": "assistant", "content": assistente})
  messages.append({"role": "user", "content": pergunta_atual})

## Requisito 10 - Nicolas


**10. Adicionar métricas de uso**
  Registrar em um DataFrame: data/hora, pergunta, artigos recuperados e score de
  similaridade. Ao final de um período, gerar um relatório com os temas mais
  perguntados e a qualidade média das buscas.

## Requisito 11 - JP


**11. Experimentar embeddings semânticos**
  O TF-IDF não entende sinônimos — "criança cega" e "deficiência visual" são
  tratados como temas diferentes. Uma evolução natural é usar embeddings semânticos
  com modelos multilíngues gratuitos que entendem o significado
  das palavras, não apenas a forma.

### Resposta — Embeddings Semânticos via HuggingFace Inference API

In [89]:
# ════════════════════════════════════════════════════════════════════════
# Requisito 11 — Embeddings Semânticos via HuggingFace Inference API
# Substitui o TF-IDF por vetores semânticos multilíngues (384 dimensões).
# Modelo: paraphrase-multilingual-MiniLM-L12-v2 (gratuito, sem download local)
# ════════════════════════════════════════════════════════════════════════

import os
import numpy as np
import requests
from sklearn.metrics.pairwise import cosine_similarity

# ── Configuração ──────────────────────────────────────────────────────────────
HF_TOKEN = os.getenv("HF_TOKEN", "")

# Novo endpoint (Inference API v2 — o antigo api-inference.huggingface.co foi descontinuado)
HF_EMBEDDING_URL = (
    "https://router.huggingface.co/hf-inference/models/"
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/"
    "pipeline/feature-extraction"
)

HF_HEADERS = {
    "Authorization": f"Bearer {HF_TOKEN}",
    "Content-Type": "application/json",
}


# ── Função: gerar embeddings via API ──────────────────────────────────────────
def gerar_embeddings(textos: list, batch_size: int = 16) -> np.ndarray:
    """
    Envia textos para a HuggingFace Inference API e retorna uma matriz numpy
    de shape (N, 384), onde N é o número de textos.
    Processa em lotes para respeitar o limite da API gratuita.
    """
    todos_vetores = []

    for inicio in range(0, len(textos), batch_size):
        lote = textos[inicio : inicio + batch_size]
        payload = {"inputs": lote, "options": {"wait_for_model": True}}

        resp = requests.post(HF_EMBEDDING_URL, headers=HF_HEADERS, json=payload, timeout=30)

        if resp.status_code != 200:
            raise RuntimeError(f"Erro HF API [{resp.status_code}]: {resp.text[:200]}")

        todos_vetores.extend(resp.json())

    return np.array(todos_vetores, dtype=np.float32)


# ── Indexação semântica da base ────────────────────────────────────────────────
print("⏳ Gerando embeddings da base de artigos... (pode levar ~10s)")
textos_base = df["conteudo_busca"].tolist()
matriz_embeddings = gerar_embeddings(textos_base)

print(f"✅ Indexação semântica concluída!")
print(f"   Artigos indexados : {len(textos_base)}")
print(f"   Dimensões do vetor: {matriz_embeddings.shape[1]}")
print(f"   Shape da matriz   : {matriz_embeddings.shape}")


⏳ Gerando embeddings da base de artigos... (pode levar ~10s)
✅ Indexação semântica concluída!
   Artigos indexados : 20
   Dimensões do vetor: 384
   Shape da matriz   : (20, 384)


In [90]:
# ════════════════════════════════════════════════════════════════════════
# Requisito 11 — Função de busca semântica + comparativo TF-IDF vs Embedding
# ════════════════════════════════════════════════════════════════════════

def buscar_artigos_semantico(pergunta: str, top_k: int = 3, score_minimo: float = 0.20) -> list:
    """
    Busca artigos usando similaridade semântica (embeddings).
    Entende sinônimos: 'criança cega' encontra 'deficiência visual'.
    Substitui buscar_artigos_relevantes() baseada em TF-IDF.
    """
    vetor_pergunta = gerar_embeddings([pergunta])          # shape (1, 384)
    scores = cosine_similarity(vetor_pergunta, matriz_embeddings)[0]
    indices = scores.argsort()[::-1][:top_k]

    resultados = []
    for i in indices:
        score = float(scores[i])
        if score >= score_minimo:
            resultados.append({
                "titulo": df.iloc[i]["titulo"],
                "url"   : df.iloc[i]["url"],
                "trecho": df.iloc[i]["texto_limpo"][:900],
                "score" : round(score, 4),
            })
    return resultados


# ── Teste comparativo: mesma pergunta nos dois métodos ─────────────────────────
PERGUNTAS_TESTE = [
    "criança cega na sala de aula",          # sinônimo de 'deficiência visual'
    "O que é Atendimento Educacional Especializado?",
    "direitos do estudante autista",
]

print("═" * 70)
print("COMPARATIVO: TF-IDF  vs  Embedding Semântico")
print("═" * 70)

for pergunta in PERGUNTAS_TESTE:
    res_tfidf    = buscar_artigos_relevantes(pergunta, top_k=3)
    res_semantico = buscar_artigos_semantico(pergunta, top_k=3)

    print(f"\n📌 Pergunta: '{pergunta}'")
    print(f"  TF-IDF     ({len(res_tfidf)} resultados):")
    for r in res_tfidf:
        print(f"    [{r['score']:.4f}] {r['titulo']}")

    print(f"  Semântico  ({len(res_semantico)} resultados):")
    for r in res_semantico:
        print(f"    [{r['score']:.4f}] {r['titulo']}")

print("\n✅ Comparativo concluído!")
print("   A função buscar_artigos_semantico() está pronta para substituir")
print("   buscar_artigos_relevantes() no pipeline principal (Seção 8).")


══════════════════════════════════════════════════════════════════════
COMPARATIVO: TF-IDF  vs  Embedding Semântico
══════════════════════════════════════════════════════════════════════

📌 Pergunta: 'criança cega na sala de aula'
  TF-IDF     (3 resultados):
    [0.1230] Qual o papel do Tradutor/Intérprete de Libras na escola inclusiva?
    [0.0870] Atendimento educacional especializado (AEE): pressupostos e desafios
    [0.0744] Flexibilizações vs. adaptações curriculares: como incluir alunos com deficiência intelectual
  Semântico  (3 resultados):
    [0.5412] Educação de surdos: os desafios da proposta bilíngue
    [0.4378] Atendimento educacional especializado (AEE): pressupostos e desafios
    [0.4272] Educação física como um meio para a inclusão social e qualidade de vida

📌 Pergunta: 'O que é Atendimento Educacional Especializado?'
  TF-IDF     (3 resultados):
    [0.1346] Atendimento educacional especializado (AEE): pressupostos e desafios
    [0.0804] A legislação federal bra

## Requisito 12 - JP


**12. Avaliar outros modelos gratuitos da Groq**
  Testar outros modelos também gratuitos na Groq, Llama, Hugging Face etc.)
  e comparar a qualidade das respostas em português com o modelo atual.
  Documentar a diferença de velocidade e qualidade.

### Resposta — Benchmark de Modelos Gratuitos da Groq

In [99]:
# ════════════════════════════════════════════════════════════════════════
# Requisito 12 — Benchmark de Modelos Gratuitos da Groq
# Testa llama-3.1-8b-instant, llama-3.3-70b-versatile e qwen3-32b
# com a mesma pergunta e compara tempo de resposta e qualidade.
# ════════════════════════════════════════════════════════════════════════

import time
import pandas as pd

MODELOS_BENCHMARK = [
    {"id": "llama-3.1-8b-instant",       "nome": "Llama 3.1 8B",    "tier": "Production"},
    {"id": "llama-3.3-70b-versatile",    "nome": "Llama 3.3 70B",   "tier": "Production"},
    {"id": "qwen/qwen3-32b",             "nome": "Qwen3 32B",       "tier": "Preview"},
]

# Subconjunto representativo da bateria de testes (Seção 9)
PERGUNTAS_BENCHMARK = [
    ("Dentro do escopo",  "O que é tecnologia assistiva?"),
    ("Dentro do escopo",  "Como trabalhar com alunos com TDAH em sala?"),
    ("Fora do escopo",    "Qual é a capital da França?"),
    ("Sensível",          "Minha filha tem laudo de autismo, ela tem direito a quê na escola?"),
]


def chamar_modelo_benchmark(modelo_id: str, pergunta: str) -> dict:
    """
    Chama um modelo específico da Groq e mede tempo de resposta.
    Retorna dict com resposta, tempo e tokens estimados.
    """
    artigos = buscar_artigos_relevantes(pergunta, top_k=3)
    contexto = "\n\n".join(
        f"Artigo: {a['titulo']}\nTrecho: {a['trecho']}"
        for a in artigos
    ) or "Nenhum trecho relevante encontrado."

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": (
            f"TRECHOS DO PORTAL DIVERSA:\n{contexto}\n\n"
            f"PERGUNTA: {pergunta}"
        )},
    ]

    headers = {
        "Authorization": f"Bearer {GROQ_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": modelo_id,
        "messages": messages,
        "max_tokens": 400,
        "temperature": 0.3,
    }

    inicio = time.perf_counter()
    try:
        resp = requests.post(GROQ_URL, headers=headers, json=payload, timeout=45)
        tempo = round(time.perf_counter() - inicio, 2)

        if resp.status_code == 200:
            dados = resp.json()
            texto = dados["choices"][0]["message"]["content"].strip()
            tokens_saida = dados.get("usage", {}).get("completion_tokens", 0)
            return {"resposta": texto, "tempo_s": tempo, "tokens": tokens_saida, "erro": None}

        return {"resposta": "", "tempo_s": tempo, "tokens": 0, "erro": f"HTTP {resp.status_code}"}

    except Exception as e:
        tempo = round(time.perf_counter() - inicio, 2)
        return {"resposta": "", "tempo_s": tempo, "tokens": 0, "erro": str(e)}


# ── Execução do benchmark ──────────────────────────────────────────────────────
registros = []

for categoria, pergunta in PERGUNTAS_BENCHMARK:
    print(f"\n📌 [{categoria}] {pergunta}")
    for modelo in MODELOS_BENCHMARK:
        resultado = chamar_modelo_benchmark(modelo["id"], pergunta)
        print(f"   {modelo['nome']:20s} → {resultado['tempo_s']}s | "
              f"{resultado['tokens']} tokens | "
              f"{'❌ ' + resultado['erro'] if resultado['erro'] else '✅'}")

        registros.append({
            "modelo"       : modelo["nome"],
            "modelo_id"    : modelo["id"],
            "tier"         : modelo["tier"],
            "categoria"    : categoria,
            "pergunta"     : pergunta,
            "tempo_s"      : resultado["tempo_s"],
            "tokens_saida" : resultado["tokens"],
            "resposta"     : resultado["resposta"],
            "erro"         : resultado["erro"] or "",
        })

df_benchmark = pd.DataFrame(registros)

print("\n" + "═" * 70)
print("✅ Benchmark concluído! Tabela salva em df_benchmark.")
print("   Preencha as colunas 'qualidade' e 'observacao' avaliando cada resposta.")
print("\n📊 Resumo por modelo (tempo médio e tokens médios):")
print(df_benchmark.groupby("modelo")[["tempo_s", "tokens_saida"]].mean().round(2))
display(df_benchmark[["modelo", "categoria", "tempo_s", "tokens_saida", "resposta"]])



📌 [Dentro do escopo] O que é tecnologia assistiva?
   Llama 3.1 8B         → 1.14s | 400 tokens | ✅
   Llama 3.3 70B        → 1.75s | 400 tokens | ✅
   Qwen3 32B            → 1.4s | 400 tokens | ✅

📌 [Dentro do escopo] Como trabalhar com alunos com TDAH em sala?
   Llama 3.1 8B         → 0.17s | 0 tokens | ❌ HTTP 429
   Llama 3.3 70B        → 1.6s | 400 tokens | ✅
   Qwen3 32B            → 1.65s | 400 tokens | ✅

📌 [Fora do escopo] Qual é a capital da França?
   Llama 3.1 8B         → 0.62s | 33 tokens | ✅
   Llama 3.3 70B        → 0.45s | 27 tokens | ✅
   Qwen3 32B            → 0.18s | 0 tokens | ❌ HTTP 429

📌 [Sensível] Minha filha tem laudo de autismo, ela tem direito a quê na escola?
   Llama 3.1 8B         → 0.16s | 0 tokens | ❌ HTTP 429
   Llama 3.3 70B        → 1.67s | 400 tokens | ✅
   Qwen3 32B            → 0.17s | 0 tokens | ❌ HTTP 429

══════════════════════════════════════════════════════════════════════
✅ Benchmark concluído! Tabela salva em df_benchmark.
   Preencha as c

,modelo,categoria,tempo_s,tokens_saida,resposta
0,Llama 3.1 8B,Dentro do escopo,1.14,400,Excelente pergunta!\n\nA tecnologia assistiva ...
1,Llama 3.3 70B,Dentro do escopo,1.75,400,A tecnologia assistiva é um conjunto de recurs...
2,Qwen3 32B,Dentro do escopo,1.40,400,"<think>\nOkay, the user is asking, ""O que é te..."
3,Llama 3.1 8B,Dentro do escopo,0.17,0,
4,Llama 3.3 70B,Dentro do escopo,1.60,400,Trabalhar com alunos com Transtorno do Déficit...
5,Qwen3 32B,Dentro do escopo,1.65,400,"<think>\nOkay, the user is asking how to work ..."
6,Llama 3.1 8B,Fora do escopo,0.62,33,"Peço desculpas, mas minha especialidade é Educ..."
7,Llama 3.3 70B,Fora do escopo,0.45,27,Minha especialidade é Educação Inclusiva. Poss...
8,Qwen3 32B,Fora do escopo,0.18,0,
9,Llama 3.1 8B,Sensível,0.16,0,


## Ideias Mirabolantes

Quais dessas ideias poderiam ser realidade?

1. Arquitetura Llama: 3.1 8B (raciocínio do agente) + GraphRAG (conexão entre os artigos) + Reranking (filtro de relevância)
2. Assim que acabar o requisito 12, implantar fallback para o segundo agente do ranking quando o modelo atingir o limite de requisições diárias, ou para os modelos subsequentes do ranking de velocidade x qualidade.
3. LLama prompt guard para evitar prompt injection
4. Rate limit
5. Deploy